# LCEL (LangChain Expression Language)

核心概念：用 `|` 操作符组合组件，支持并行、透传、自定义 Lambda。

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 1. 基础链

In [ ]:
chain = ChatPromptTemplate.from_template("给主题起三个好标题：{topic}") | llm | StrOutputParser()
print(chain.invoke({"topic": "AI 编程助手"}))

### 2. RunnablePassthrough 透传

In [ ]:
chain = (
    {"topic": RunnablePassthrough()}
    | ChatPromptTemplate.from_template("写一句关于{topic}的话")
    | llm | StrOutputParser()
)
print(chain.invoke("人工智能"))

### 3. 并行执行

同时执行多个链，结果合并返回。

In [ ]:
chain1 = ChatPromptTemplate.from_template("用一句话总结：{topic}") | llm | StrOutputParser()
chain2 = ChatPromptTemplate.from_template("列出{topic}的三个关键点") | llm | StrOutputParser()

parallel = RunnableParallel(summary=chain1, keypoints=chain2)
result = parallel.invoke({"topic": "RAG 技术"})
print("总结:", result["summary"][:100])
print("要点:", result["keypoints"][:200])

### 4. RunnableLambda 注入自定义函数

In [ ]:
def word_count(text: str) -> dict:
    return {"chars": len(text), "words": len(text.split())}

chain = (
    ChatPromptTemplate.from_template("写一句关于{topic}的话")
    | llm | StrOutputParser()
    | RunnableLambda(word_count)
)
print(chain.invoke({"topic": "机器学习"}))

### 5. 批处理

一次性处理多个输入。

In [ ]:
chain = ChatPromptTemplate.from_template("介绍{topic}（一句话）") | llm | StrOutputParser()
results = chain.batch([{"topic": "Python"}, {"topic": "Rust"}])
for i, r in enumerate(results, 1):
    print(f"{i}. {r}")